# Reconhecimento Facial com FaceNet: Avaliação de Desempenho e Fine-Tuning

**Projeto de Inteligência Artificial — Ciência da Computação**

## 1. Introdução

O reconhecimento facial é uma aplicação da Inteligência Artificial que utiliza técnicas de aprendizado profundo para analisar características de rostos humanos.

Neste projeto, utilizamos o **FaceNet**, uma rede neural capaz de transformar imagens faciais em representações numéricas denominadas *embeddings*. Essas representações permitem comparar rostos utilizando medidas de similaridade e distância.

O trabalho foi dividido em dois experimentos:

- **Experimento 1 — LFW:** avaliação do FaceNet pré-treinado, utilizando similaridade do cosseno e distância euclidiana, com validação cruzada em 10 folds.
- **Experimento 2 — CelebA:** aplicação de fine-tuning em um subconjunto de imagens e comparação do desempenho do modelo original com o modelo ajustado.

O objetivo é avaliar o desempenho do FaceNet na verificação facial e investigar se o treinamento adicional melhora sua capacidade de distinguir rostos de pessoas diferentes.

Se os embeddings forem suficientemente semelhantes, o sistema poderá considerar que as duas fotos pertencem à mesma pessoa. Para decidir isso, precisamos escolher e avaliar um limite de similaridade.

In [ ]:
#Biblioteca com ferramentas para reconhecimento facial
#O FaceNet é uma rede neural que transforma um rosto em uma representação numérica chamada embedding.
%pip install --no-deps facenet-pytorch

**Configuração do Ambiente:**

*  Torch
*  Numpy
*  facenet-pytorch - disponibiliza os modelos FaceNet (transforma os rostos em veotres- embeddings)
*  MTCNN: detecta e prepara as orostos nas imagens

Foi utilizado GPU



In [ ]:
import torch
import numpy

print("PyTorch:", torch.__version__)
print("NumPy:", numpy.__version__)
print("GPU disponível:", torch.cuda.is_available())

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Dispositivo escolhido:", device)

In [ ]:
import torchvision

print("TorchVision:", torchvision.__version__)

In [ ]:
#MTCNN — Localizar o rosto
#modelo: o FaceNet pré-treinado.
#InceptionResnetV1 — Extrair características
#É a arquitetura de rede neural disponibilizada pela biblioteca para gerar embeddings faciais. Podemos carregá-la com pesos pré-treinados.
from facenet_pytorch import MTCNN, InceptionResnetV1

print("FaceNet importado com sucesso!")


começar com um modelo pré-treinado, medir seu desempenho e depois testar outras configurações para comparar os resultados.
InceptionResnetV1:É a arquitetura da rede neural que consegue extrair características de rostos/embedding faciais
- Os embeddings "transformam" características do rosto em vetores numéricos


In [ ]:
model = InceptionResnetV1(
    pretrained="vggface2" #pesos de um modelo que já foi treinado utilizando o conjunto de dados VGGFace2.
).eval().to(device)#Coloca o modelo em modo de avaliação, Envia o modelo para o dispositivo que configuramos anteriormente.

print("Modelo carregado!")

rede neural pronta para extrair características de rostos
- Avaliei o desempenho do FaceNet pré-treinado para verificar se duas imagens pertencem ao mesmo rosto
- Foi utlizado duas medias de comparação:


1.   Similiaridade do cosseno
2.   Distancia euclidiana



In [ ]:
detector = MTCNN(
    image_size=160,
    margin=0,
    device=device
)#detector de rostos e guardando-o na variável detector.

print("Detector carregado!")

AO ter a fotografia -> MTCNN detecta o rosto -> FaceNet transforma o rosto em representacao numerica

In [ ]:
from PIL import Image #ferramenta para abrir imagens
import requests #endereco da internet
from io import BytesIO #tratar os dados baixados como um arquivo na memória.

url = "https://raw.githubusercontent.com/ageitgey/face_recognition/master/examples/obama.jpg"
#pega a imagem
resposta = requests.get(url, timeout=20)
#informa se teve erro
resposta.raise_for_status()
#abre a imagem
imagem = Image.open(BytesIO(resposta.content)).convert("RGB")
display(imagem) #mostra a fotografia

In [ ]:
#mtcnn para encontrar o rosto
rosto = detector(imagem)
#Mostra o formato do tensor produzido.
print(rosto.shape)

O MTCNN recebeu a foto, localizou o rosto e transformou em um tensor

In [ ]:
import matplotlib.pyplot as plt
#biblioteca de visuzalizacao
#visuzalizar o rosto detecteado pelo MTCNN
#organiza as dimensao (cainais, altura, largura)->(altura,largura, canais)

plt.imshow(rosto.permute(1, 2, 0).cpu().numpy() * 0.5 + 0.5)
#exibe imagem e esconde eiox
plt.axis("off")
plt.show()

detectar, recortar e preparar um rosto para a rede neural.
entregar o rosto recortado a esse modelo.
torch.no_grad esta sendo usado pq nao é preciso calcular gradiente, uma vez que nao esta havendo treinamento

In [ ]:
with torch.no_grad():
  #a rede neural processa o rosto e produz um embedding.
    embedding = model(rosto.unsqueeze(0).to(device))

print("Formato do embedding:", embedding.shape)
#resultado: 1 rosto 512 representacao numerica
#512 carcteristicas aprendidas do rosto
#uma linha com 512 numeros

In [ ]:
print(embedding[0, :10])
#o primeiro rosto e os 10 primeiros numeros do embedding

gerar um embedding para cada rosto e depois medir a semelhança entre essas representações.
Agora sera feito comparacao
o embedding na primeira imagem esta guardado na variavel embedding

In [ ]:
url2 = "https://raw.githubusercontent.com/ageitgey/face_recognition/master/examples/obama2.jpg"

resposta2 = requests.get(url2, timeout=20)
resposta2.raise_for_status()

imagem2 = Image.open(
    BytesIO(resposta2.content)
).convert("RGB")

display(imagem2)

Verficar se imagens com a mesma pessoa gera embeddings semelhantes

In [ ]:
rosto2 = detector(imagem2)

if rosto2 is not None:
    with torch.no_grad():
        embedding2 = model(rosto2.unsqueeze(0).to(device))
#model:passa o rosto pelo FAceNet e produz embedding
    print("Formato do segundo embedding:", embedding2.shape)
else:
    print("Nenhum rosto foi detectado.")

In [ ]:
print(embedding2[0, :10])

In [ ]:
#funcoes matematicas
import torch.nn.functional as F
#compara direcao de vetores
similaridade = F.cosine_similarity(
    embedding,
    embedding2,
    dim=1
)
#Transforma o resultado, que é um tensor com um único valor, em um número Python para facilitar a leitura.
#varia de -1 e 1, Quanto mais próximo de 1, mais semelhantes são as direções dos embeddings.
print("Similaridade:", similaridade.item())

### Experimento 1 — Avaliação inicial do FaceNet

**Objetivo:** avaliar a capacidade de um modelo facial pré-treinado de distinguir pares de imagens da mesma pessoa e de pessoas diferentes.

**Modelo:** InceptionResnetV1, pré-treinado com VGGFace2.

**Detecção e alinhamento:** MTCNN, com imagens de 160 × 160 pixels.

**Métrica de comparação:** similaridade de cosseno.

**Dataset candidato:** Labeled Faces in the Wild (LFW), sujeito à verificação das condições de uso e do protocolo de avaliação.

**Hipótese:** pares da mesma pessoa apresentarão, em média, similaridade maior do que pares de pessoas diferentes.

**Estratégia:** não realizar fine-tuning neste primeiro experimento. Separar os dados destinados à escolha do limiar daqueles reservados à avaliação final.

**Resultados:** a preencher após os testes.

In [ ]:
#Quanto menor, mais próximos os embeddings
#descobrir o quanto dois rostos são parecidos matematicamente.
#sao pontos em um grafico, quanto mais proximo os pontos mais semlhante
#entao para ver a semelhanca pode ser usado como metricas- cosseno e distancia euclidiana
distancia = torch.dist(
    embedding[0],
    embedding2[0]
)

print("Distância euclidiana:", distancia.item())

Semelhanca dos cossenos=0.84 (perto de 1)
distancia=0.55 (baixa distanica)
Mas ainda não podemos afirmar que o sistema reconheceu corretamente a pessoa
Agora sera adicionado uma pessoa diferente para fazer a comparacao com as fotos 1 e 2 e comprovar que a semelhanca sera mais distante de 1 e a distancia sera maior

In [ ]:
url3 = "https://raw.githubusercontent.com/ageitgey/face_recognition/master/examples/biden.jpg"

resposta3 = requests.get(url3, timeout=20)
resposta3.raise_for_status()

imagem3 = Image.open(
    BytesIO(resposta3.content)
).convert("RGB")

display(imagem3)

In [ ]:
rosto3 = detector(imagem3)

if rosto3 is not None:
    with torch.no_grad():
        embedding3 = model(rosto3.unsqueeze(0).to(device))
#model:passa o rosto pelo FAceNet e produz embedding
    print("Formato do segundo embedding:", embedding3.shape)
else:
    print("Nenhum rosto foi detectado.")
    rosto3 = detector(imagem3)



In [ ]:
print(embedding3[0, :10])

In [ ]:
#funcoes matematicas
import torch.nn.functional as F
#compara direcao de vetores
similaridade = F.cosine_similarity(
    embedding,
    embedding3,
    dim=1
)
#compraracao 1 e 3
#varia de -1 e 1, Quanto mais próximo de 1, mais semelhantes são as direções dos embeddings.
print("Similaridade:", similaridade.item())

In [ ]:
#funcoes matematicas
import torch.nn.functional as F
#compara direcao de vetores
similaridade = F.cosine_similarity(
    embedding2,
    embedding3,
    dim=1
)
#comparaacao 2 e 3
#varia de -1 e 1, Quanto mais próximo de 1, mais semelhantes são as direções dos embeddings.
print("Similaridade:", similaridade.item())

In [ ]:
#Quanto menor, mais próximos os embeddings
#descobrir o quanto dois rostos são parecidos matematicamente.
#sao pontos em um grafico, quanto mais proximo os pontos mais semlhante
#entao para ver a semelhanca pode ser usado como metricas- cosseno e distancia euclidiana
distancia = torch.dist(
    embedding[0],
    embedding3[0]
)
#comparacao de distancia 1 e 3
print("Distância euclidiana:", distancia.item())

In [ ]:

#comparcao de distancia 2 e 3
distancia = torch.dist(
    embedding2[0],
    embedding3[0]
)

print("Distância euclidiana:", distancia.item())

In [ ]:
#criar uma tabela com os resultados
#DataFrame constroi a tabela
#biblioteca pandas do Python: usada para organizar, manipular e analisar dados.
import pandas as pd

resultados = pd.DataFrame({
    "Comparacao": [
        "Foto 1 x Foto 2",
        "Foto 1 x Foto 3",
        "Foto 2 x Foto 3"
    ],
    "Mesma_pessoa": [
        True,
        False,
        False
    ],
    "Similaridade": [
        0.84,
        -0.0288,
        0.0724
    ],
    "Distancia": [
        0.55,
        1.4344,
        1.3620
    ]
})

display(resultados)

Fiz tudo manualmente, mas isso é inviavel fazer com muitas imagens, entao vou fazer um loop

In [ ]:
fotos = [
    ("Foto 1", embedding),
    ("Foto 2", embedding2),
    ("Foto 3", embedding3)
]

for nome, vetor in fotos:
    print(nome, vetor.shape)

In [ ]:
from itertools import combinations

resultados = []

for foto1, foto2 in combinations(fotos, 2):
    nome1, vetor1 = foto1
    nome2, vetor2 = foto2

    similaridade = F.cosine_similarity(
        vetor1, vetor2, dim=1
    ).item()

    distancia = torch.dist(
        vetor1[0], vetor2[0]
    ).item()

    resultados.append({
        "Comparacao": f"{nome1} x {nome2}",
        "Similaridade": similaridade,
        "Distancia": distancia
    })

tabela = pd.DataFrame(resultados)
display(tabela)

 criar um threshold experimental para delimitar um limite para definir se é ou não a mesma pessoa

threshold = 0.70
tabela["Predicao_mesma_pessoa"] = (
    tabela["Similaridade"] >= threshold
)

display(tabela)

In [ ]:
# Definir o limiar de similaridade
threshold = 0.70

# Criar a coluna de previsão
tabela["Predicao_mesma_pessoa"] = (
    tabela["Similaridade"] >= threshold
)

# Respostas reais dos pares
tabela["Mesma_pessoa_real"] = [
    True,
    False,
    False
]

# Verificar os acertos
tabela["Acertou"] = (
    tabela["Predicao_mesma_pessoa"]
    == tabela["Mesma_pessoa_real"]
)

# Calcular acurácia
acuracia = tabela["Acertou"].mean()

print(f"Acurácia: {acuracia * 100:.2f}%")

display(tabela)

In [ ]:
#De todos os pares que realmente são da mesma pessoa, quantos o sistema conseguiu identificar corretamente
vp =(
    (tabela["Mesma_pessoa_real"]==True) &
    (tabela["Predicao_mesma_pessoa"]==True)
).sum()
fn = (
    (tabela["Mesma_pessoa_real"] == True) &
    (tabela["Predicao_mesma_pessoa"] == False)
).sum()

recall = vp / (vp + fn) if (vp + fn) > 0 else 0

print("Verdadeiros positivos:", vp)
print("Falsos negativos:", fn)
print(f"Recall: {recall * 100:.2f}%")

In [ ]:
vp = (
    (tabela["Mesma_pessoa_real"] == True) &
    (tabela["Predicao_mesma_pessoa"] == True)
).sum()

fp = (
    (tabela["Mesma_pessoa_real"] == False) &
    (tabela["Predicao_mesma_pessoa"] == True)
).sum()

precisao = vp / (vp + fp) if (vp + fp) > 0 else 0

print("Verdadeiros positivos:", vp)
print("Falsos positivos:", fp)
print(f"Precisão: {precisao * 100:.2f}%")

In [ ]:
#O F1-score ajuda a avaliar o equilibrio entre precisao e recall
f1 = (
    2 * (precisao * recall) / (precisao + recall)
    if (precisao + recall) > 0
    else 0
)

print(f"F1-score: {f1:.2f}")
print(f"F1-score em porcentagem: {f1 * 100:.2f}%")

#FAR— False Acceptance Rate
Taxa de falsa aceitação. Mede a proporção de tentativas de pessoas diferentes que o sistema aceita incorretamente como correspondentes.
Exemplo: o sistema confunde uma pessoa desconhecida com alguém autorizado.




# FRR — False Rejection Rate
Taxa de falsa rejeição. Mede a proporção de tentativas legítimas que o sistema rejeita incorretamente.
Exemplo: uma pessoa autorizada tenta entrar, mas o sistema não reconhece seu rosto.

In [ ]:
vn = (
    (tabela["Mesma_pessoa_real"] == False) &
    (tabela["Predicao_mesma_pessoa"] == False)
).sum()

far = fp / (fp + vn) if (fp + vn) > 0 else 0
frr = fn / (fn + vp) if (fn + vp) > 0 else 0

print("Verdadeiros negativos:", vn)
print(f"FAR: {far * 100:.2f}%")
print(f"FRR: {frr * 100:.2f}%")

In [ ]:
metricas = pd.DataFrame({
    "Metrica": [
        "Acuracia",
        "Precisao",
        "Recall",
        "F1-score",
        "FAR",
        "FRR"
    ],
    "Resultado (%)": [
        acuracia * 100,
        precisao * 100,
        recall * 100,
        f1 * 100,
        far * 100,
        frr * 100
    ]
})

display(metricas)

Configuração do PyTorch e GPU ok

Detecção de rostos com MTCNN ok

Geração de embeddings com FaceNet ok

Comparações com cosseno e distância euclidiana ok

Cálculo das seis métricas ok


Dataset maior e divisão dos dados

Escolha do threshold por validação ok

Experimentos de fine-tuning

Comparação e análise dos resultados finais

O LFW (Labeled Faces in the Wild) é um banco de dados público de imagens de rostos criado por pesquisadores

#Avaliação do FaceNet no LFW
Aplicar o FaceNet (pré-treinado) em LFW para verificar se duas imagens sao da mesma pessoa
-FaceNet utiliza como comparador a similaridade do cosseno e a distancia euclidiana
-As métricas de avaliação foram utilizados:


1.   Meio de Acurácia
2.   Precisão
3.   Recall
4.   F1-score
5.   Taxa de falsa-aceitaçao FAR
6.   Taxa de falsa rejeição FRR



In [ ]:
from sklearn.datasets import fetch_lfw_pairs
#permite carregar pares de fotografias do LFW.
print("Importação realizada com sucesso!")

A função que importamos permite acessar dois subconjuntos:
- train: pares destinados ao desenvolvimento e à escolha de configurações, como nosso threshold.
- test: pares reservados para a avaliação final.

Imagine que temos vários pares de fotografias e os dividimos em 10 grupos.

In [ ]:

lfw = fetch_lfw_pairs(
    subset="10_folds",
    resize=1.0,
    download_if_missing=True
)

print("Quantidade de pares:", len(lfw.target))
print("Formato das imagens:", lfw.pairs.shape)

In [ ]:
import matplotlib.pyplot as plt
#o primeiro par do dataset
par = lfw.pairs[0]

plt.figure(figsize=(8, 4))
#as duas fotografias do par
plt.subplot(1, 2, 1)
plt.imshow(par[0], cmap="gray")
plt.title("Foto 1")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(par[1], cmap="gray")
plt.title("Foto 2")
plt.axis("off")

plt.show()
#informa a resposta
#1 -> mesma pessoa
#0 -> não são a mesmo pessoa
print("Rótulo do par:", lfw.target[0])

FaceNet recebe imagens com o formato:
[3, 160, 160]
Isso significa:
- 3 canais de cores (RGB).
- 160 pixels de altura.
- 160 pixels de largura.
LFW carregadas pelo scikit-learn são menores e estão em tons de cinza.
Portanto, não podemos simplesmente colocar essas imagens diretamente no FaceNet. Primeiro precisaremos adaptá-las, preservando o pré-processamento esperado pelo modelo.


*   Precisamos transformar as imagens para que o FaceNet consiga processá-las.
Um detalhe: aumentar uma imagem de 125 × 94 para 160 × 160 não cria detalhes novos. Essa é uma limitação que vamos documentar no experimento.





In [ ]:
print("Formato do par:", par.shape)

print("Formato da primeira foto:", par[0].shape)

print("Formato da segunda foto:", par[1].shape)

print("Tipo dos dados:", par.dtype)
#verificar a escola dos pixels
#se tiver entre 0 e 1 esta normalizada
print("Menor valor:", par[0].min())
print("Maior valor:", par[0].max())

Converter a imagem de tons de cinza para três canais.



Redimensionar para 160 × 160 pixels.



Padronizar os valores dos pixels.

In [ ]:
from torchvision.transforms import functional as TF

# Converte a imagem do LFW para tensor
foto_tensor = torch.from_numpy(par[0].copy())

# Adiciona a dimensão do canal: [1, 125, 94]
foto_tensor = foto_tensor.unsqueeze(0)
#torna a imagem compativel com rgb, mas nn recupera cor orginial
# Repete o canal cinza 3 vezes: [3, 125, 94]
foto_tensor = foto_tensor.repeat(3, 1, 1)

# Ajusta o tamanho para FaceNet
foto_tensor = TF.resize(foto_tensor, [160, 160])

# Padroniza os pixels para a escala usada pelo FaceNet
foto_tensor = (foto_tensor * 255 - 127.5) / 128.0

print("Formato final:", foto_tensor.shape)
print("Menor valor:", foto_tensor.min().item())
print("Maior valor:", foto_tensor.max().item())

In [ ]:
from torchvision.transforms import functional as TF

# Converte a imagem do LFW para tensor
foto2_tensor = torch.from_numpy(par[1].copy())

# Adiciona a dimensão do canal: [1, 125, 94]
foto2_tensor = foto2_tensor.unsqueeze(0)
#torna a imagem compativel com rgb, mas nn recupera cor orginial
# Repete o canal cinza 3 vezes: [3, 125, 94]
foto2_tensor = foto2_tensor.repeat(3, 1, 1)

# Ajusta o tamanho para FaceNet
foto2_tensor = TF.resize(foto2_tensor, [160, 160])

# Padroniza os pixels para a escala usada pelo FaceNet
foto2_tensor = (foto2_tensor * 255 - 127.5) / 128.0

print("Formato final:", foto2_tensor.shape)
print("Menor valor:", foto2_tensor.min().item())
print("Maior valor:", foto2_tensor.max().item())

In [ ]:

with torch.no_grad():

    embedding_lfw = model(
        foto_tensor.unsqueeze(0).to(device)
    )
    embedding_lfw2 = model(
        foto2_tensor.unsqueeze(0).to(device)
    )

similaridade_lfw = F.cosine_similarity(
    embedding_lfw,
    embedding_lfw2,
    dim=1
).item()

distancia_lfw = torch.dist(
    embedding_lfw[0],
    embedding_lfw2[0]
).item()

print(f"Similaridade: {similaridade_lfw:.4f}")
print(f"Distância euclidiana: {distancia_lfw:.4f}")
print("Mesma pessoa (real):", bool(lfw.target[0]))

In [ ]:
def preparar_imagem(imagem):
    tensor = torch.from_numpy(imagem.copy())

    tensor = tensor.unsqueeze(0)

    tensor = tensor.repeat(3, 1, 1)

    tensor = TF.resize(tensor, [160, 160])

    tensor = (tensor * 255 - 127.5) / 128.0

    return tensor
teste = preparar_imagem(par[0])

print("Formato:", teste.shape)

In [ ]:
def gerar_embedding(imagem):
    tensor = preparar_imagem(imagem)

    tensor = tensor.unsqueeze(0).to(device)

    with torch.no_grad():
        embedding = model(tensor)

    return embedding

In [ ]:
embedding_teste = gerar_embedding(par[0])

print("Formato:", embedding_teste.shape)

In [ ]:
def comparar_rostos(imagem1, imagem2):
    embedding1 = gerar_embedding(imagem1)
    embedding2 = gerar_embedding(imagem2)

    similaridade = F.cosine_similarity(
        embedding1, embedding2, dim=1
    ).item()

    distancia = torch.dist(
        embedding1[0], embedding2[0]
    ).item()
#funcao que calcula similaridade e distancia juntos
    return similaridade, distancia

In [ ]:
similaridade, distancia = comparar_rostos(
    lfw.pairs[0][0],
    lfw.pairs[0][1]
)

print(f"Similaridade: {similaridade:.4f}")
print(f"Distância: {distancia:.4f}")

In [ ]:
resultados_lfw = []

for i in range(10):
    imagem1 = lfw.pairs[i][0]
    imagem2 = lfw.pairs[i][1]
    #Selecionam as duas fotografias do par atual.

    similaridade, distancia = comparar_rostos(
        imagem1, imagem2
    )
    #Ela prepara as imagens, gera os embeddings e calcula as duas métricas.

    resultados_lfw.append({
        "Par": i,
        "Mesma_pessoa_real": bool(lfw.target[i]),
        "Similaridade": similaridade,
        "Distancia": distancia
    })

tabela_lfw = pd.DataFrame(resultados_lfw)

display(tabela_lfw)

In [ ]:
threshold = 0.70

tabela_lfw["Predicao_mesma_pessoa"] = (
    tabela_lfw["Similaridade"] >= threshold
)

display(tabela_lfw)
#Coluna	Significado
#Mesma_pessoa_real	A resposta correta fornecida pelo LFW
#Predicao_mesma_pessoa	A resposta calculada pelo nosso sistema

In [ ]:
real = tabela_lfw["Mesma_pessoa_real"]
predicao = tabela_lfw["Predicao_mesma_pessoa"]

vp = (real & predicao).sum()

fp = (~real & predicao).sum()

vn = (~real & ~predicao).sum()

fn = (real & ~predicao).sum()

print("VP:", vp)
print("FP:", fp)
print("VN:", vn)
print("FN:", fn)

print("Total:", vp + fp + vn + fn)

In [ ]:
acuracia_lfw = (vp + vn) / (vp + fp + vn + fn)

precisao_lfw = vp / (vp + fp) if (vp + fp) > 0 else 0

recall_lfw = vp / (vp + fn) if (vp + fn) > 0 else 0

f1_lfw = (
    2 * precisao_lfw * recall_lfw / (precisao_lfw + recall_lfw)
    if (precisao_lfw + recall_lfw) > 0 else 0
)

far_lfw = fp / (fp + vn) if (fp + vn) > 0 else 0

frr_lfw = fn / (fn + vp) if (fn + vp) > 0 else 0
#as seis métricas usando os 10 pares do LFW, em vez das três fotografias anteriores.
metricas_lfw = pd.DataFrame({
    "Metrica": ["Acuracia", "Precisao", "Recall", "F1", "FAR", "FRR"],
    "Resultado (%)": [
        acuracia_lfw * 100,
        precisao_lfw * 100,
        recall_lfw * 100,
        f1_lfw * 100,
        far_lfw * 100,
        frr_lfw * 100
    ]
})

display(metricas_lfw)

In [ ]:
print("Quantidade de pares:", len(tabela_lfw))

print("Primeiro par:")
#É uma forma de selecionar a primeira linha de uma tabela Pandas.
print(tabela_lfw.iloc[0])

print("\nContagem dos resultados:")
print("VP:", vp)
print("FP:", fp)
print("VN:", vn)
print("FN:", fn)

print("Total:", vp + fp + vn + fn)

In [ ]:
import numpy as np

far_lfw = (
    fp / (fp + vn)
    if (fp + vn) > 0
    else np.nan
)

print("FAR:", far_lfw)

In [ ]:
import numpy as np

indices_negativos = np.where(lfw.target == 0)[0]

print("Quantidade de pares negativos:", len(indices_negativos))
print("Primeiros índices negativos:", indices_negativos[:10])

In [ ]:
resultados_negativos = []

for i in indices_negativos[:10]:
    imagem1 = lfw.pairs[i][0]
    imagem2 = lfw.pairs[i][1]

    similaridade, distancia = comparar_rostos(
        imagem1, imagem2
    )

    resultados_negativos.append({
        "Par": i,
        "Mesma_pessoa_real": bool(lfw.target[i]),
        "Similaridade": similaridade,
        "Distancia": distancia
    })

tabela_negativos = pd.DataFrame(resultados_negativos)

display(tabela_negativos)

In [ ]:
print("Total de pares:", len(lfw.target))

print("Pares por fold:", len(lfw.target) // 10)

for i in range(10):
    inicio = i * 600
    fim = inicio + 600

    print(f"Fold {i+1}: índices {inicio} até {fim-1}")

In [ ]:
from tqdm.auto import tqdm
#serve para mostrar uma barra de progresso enquanto um loop está sendo executado.

In [ ]:
def avaliar_pares(lfw, quantidade):
    resultados = []

    for i in tqdm(range(quantidade)):
        imagem1 = lfw.pairs[i][0]
        imagem2 = lfw.pairs[i][1]

        similaridade, distancia = comparar_rostos(
            imagem1, imagem2
        )

        resultados.append({
            "Par": i,
            "Mesma_pessoa_real": bool(lfw.target[i]),
            "Similaridade": similaridade,
            "Distancia": distancia
        })

    return pd.DataFrame(resultados)

In [ ]:
teste_30 = avaliar_pares(lfw, 30)

print("Pares processados:", len(teste_30))

In [ ]:
imagens = [
    lfw.pairs[0][0],
    lfw.pairs[0][1],
    lfw.pairs[1][0],
    lfw.pairs[1][1]
]

tensores = [preparar_imagem(img) for img in imagens]

lote = torch.stack(tensores).to(device)

print("Formato do lote:", lote.shape)

with torch.no_grad():
    embeddings_lote = model(lote)

print("Formato dos embeddings:", embeddings_lote.shape)
#tensores: guarda as quatro imagens já preparadas.
#torch.stack(tensores): junta as quatro imagens em um único tensor.
#modelo(lote): envia as quatro imagens juntas para o FaceNet.


In [ ]:
def gerar_embeddings_lote(imagens, tamanho_lote=32):
  #vai processar ate 32 imagens de uma so vez
    todos_embeddings = []

    for inicio in tqdm(range(0, len(imagens), tamanho_lote)):

        fim = inicio + tamanho_lote

        imagens_lote = imagens[inicio:fim]

        tensores = [
            preparar_imagem(img)
            for img in imagens_lote
        ]

        lote = torch.stack(tensores).to(device)


        with torch.no_grad():
            embeddings = model(lote)

        todos_embeddings.append(embeddings.cpu())

    return torch.cat(todos_embeddings, dim=0)
    #Junta os embeddings de todos os lotes em um único tensor.

In [ ]:
imagens_teste = []

for i in range(5):
    imagens_teste.append(lfw.pairs[i][0])
    imagens_teste.append(lfw.pairs[i][1])

embeddings_teste = gerar_embeddings_lote(
    imagens_teste,
    tamanho_lote=4
)

print("Formato final:", embeddings_teste.shape)

In [ ]:
embedding1 = embeddings_teste[0::2] #pega os embeddings nas posições 0, 2, 4, 6 e 8
embedding2 = embeddings_teste[1::2]#pega os embeddings nas posicoes 1,3,5,7,9
#começar usando os 5 pares que acabamos de processar.
similaridades = F.cosine_similarity(
    embedding1,
    embedding2,
    dim=1
)
#O F.cosine_similarity() calcula a similaridade de cada par, enquanto torch.norm() calcula a distância euclidiana.
distancias = torch.norm(
    embedding1 - embedding2,
    dim=1
)

print("Similaridades:", similaridades)
print("Distâncias:", distancias)

print("Quantidade de pares:", len(similaridades))

In [ ]:
def avaliar_pares_em_lotes(lfw, quantidade):
    imagens = []
#Agora, avaliar_pares_em_lotes() utiliza gerar_embeddings_lote(), que envia várias imagens juntas para a GPU.
    for i in range(quantidade):
        imagens.append(lfw.pairs[i][0])
        imagens.append(lfw.pairs[i][1])

    embeddings = gerar_embeddings_lote(
        imagens,
        tamanho_lote=32
    )

    embedding1 = embeddings[0::2]
    embedding2 = embeddings[1::2]

    similaridades = F.cosine_similarity(
        embedding1, embedding2, dim=1
    )

    distancias = torch.norm(
        embedding1 - embedding2, dim=1
    )

    resultados = pd.DataFrame({
        "Par": range(quantidade),
        "Mesma_pessoa_real": lfw.target[:quantidade].astype(bool),
        "Similaridade": similaridades.numpy(),
        "Distancia": distancias.numpy()
    })

    return resultados

In [ ]:
#teste
teste_lotes_30 = avaliar_pares_em_lotes(lfw, 30)

print("Quantidade de pares:", len(teste_lotes_30))

display(teste_lotes_30.head())

### Validação do processamento em lotes

Para otimizar o processamento das imagens, foi implementada
uma função capaz de gerar embeddings em lotes.

A fim de verificar a equivalência entre as abordagens,
foram processados 30 pares de imagens utilizando tanto
o processamento individual quanto o processamento em lotes.

A maior diferença observada entre as similaridades foi
de aproximadamente 3,58 × 10⁻⁷, e os resultados foram
considerados equivalentes pela função `np.allclose`,
com tolerância absoluta de 10⁻⁵.

Esse resultado indica que o processamento em lotes
preservou a precisão numérica das similaridades
nesse conjunto de teste.

In [ ]:
diferencas = abs(
    teste_30["Similaridade"] -
    teste_lotes_30["Similaridade"]
)

print("Maior diferença:", diferencas.max())

print(
    "Resultados equivalentes:",
    np.allclose(
        teste_30["Similaridade"],
        teste_lotes_30["Similaridade"],
        atol=1e-5
    )
)

In [ ]:
tabela_completa = avaliar_pares_em_lotes(
    lfw,
    len(lfw.target)#informa à função que queremos processar todos os pares.
)

print("Total de pares:", len(tabela_completa))

print(
    "Pares da mesma pessoa:",
    tabela_completa["Mesma_pessoa_real"].sum()#guarda as similaridades e distâncias dos 6.000 pares.
)

print(
    "Pares de pessoas diferentes:",
    (~tabela_completa["Mesma_pessoa_real"]).sum()#conta quantos pares são da mesma pessoa.
)

display(tabela_completa.head())

In [ ]:
tabela_completa["Fold"] = (
    tabela_completa["Par"] // 600
) + 1

display(tabela_completa.head())

print(
    tabela_completa["Fold"]
    .value_counts()
    .sort_index()
)
#Lembra que o LFW possui 10 folds, cada um com 600 pares?
#Vamos adicionar uma coluna à nossa tabela indicando a qual fold cada par pertence.

In [ ]:
fold_teste = 1

dados_desenvolvimento = tabela_completa[
    tabela_completa["Fold"] != fold_teste
]
#selecione todos os pares cujo fold seja diferente de 1.
dados_teste = tabela_completa[
    tabela_completa["Fold"] == fold_teste
]
#selecione somente os pares do fold 1.
print("Pares de desenvolvimento:", len(dados_desenvolvimento))
print("Pares de teste:", len(dados_teste))

Descobrir qual melhor thresholds

In [ ]:
thresholds = np.arange(0.0, 1.01, 0.01)

resultados_thresholds = []

for threshold in thresholds:

    real = dados_desenvolvimento["Mesma_pessoa_real"]

    predicao = (
        dados_desenvolvimento["Similaridade"] >= threshold
    )

    acuracia = (real == predicao).mean()

    resultados_thresholds.append({
        "Threshold": round(threshold, 2),
        "Acuracia": acuracia
    })

tabela_thresholds = pd.DataFrame(resultados_thresholds)

display(tabela_thresholds.head())

In [ ]:
melhor_indice = tabela_thresholds["Acuracia"].idxmax()

melhor_resultado = tabela_thresholds.loc[melhor_indice]

melhor_threshold = melhor_resultado["Threshold"]

print("Melhor threshold:", melhor_threshold)

print(
    "Melhor acurácia:",
    round(melhor_resultado["Acuracia"] * 100, 2),
    "%"
)

In [ ]:
real_teste = dados_teste["Mesma_pessoa_real"]

predicao_teste = (
    dados_teste["Similaridade"] >= melhor_threshold
)

acuracia_teste = (real_teste == predicao_teste).mean()

print("Threshold utilizado:", melhor_threshold)

print(
    "Acurácia no fold 1:",
    round(acuracia_teste * 100, 2),
    "%"
)

print("Quantidade de pares testados:", len(dados_teste))

In [ ]:
def encontrar_melhor_threshold(dados):
    thresholds = np.arange(0.0, 1.01, 0.01)

    melhor_threshold = None
    melhor_acuracia = -1

    for threshold in thresholds:
        real = dados["Mesma_pessoa_real"]

        predicao = dados["Similaridade"] >= threshold

        acuracia = (real == predicao).mean()

        if acuracia > melhor_acuracia:
            melhor_acuracia = acuracia
            melhor_threshold = threshold

    return melhor_threshold, melhor_acuracia

In [ ]:
threshold_encontrado, acuracia_encontrada = (
    encontrar_melhor_threshold(dados_desenvolvimento)
)

print("Threshold:", round(threshold_encontrado, 2))
print("Acurácia:", round(acuracia_encontrada * 100, 2), "%")

In [ ]:
resultados_folds = []

for fold in range(1, 11):

    # 1. Separar desenvolvimento e teste
    desenvolvimento = tabela_completa[
        tabela_completa["Fold"] != fold
    ]

    teste = tabela_completa[
        tabela_completa["Fold"] == fold
    ]

    # 2. Encontrar o melhor threshold
    threshold, acuracia_dev = encontrar_melhor_threshold(
        desenvolvimento
    )

    # 3. Avaliar no fold reservado para teste
    real = teste["Mesma_pessoa_real"]
    predicao = teste["Similaridade"] >= threshold

    acuracia_teste = (real == predicao).mean()

    # 4. Guardar os resultados
    resultados_folds.append({
        "Fold": fold,
        "Threshold": round(threshold, 2),
        "Acuracia_teste": acuracia_teste * 100
    })

tabela_folds = pd.DataFrame(resultados_folds)

display(tabela_folds)

In [ ]:
media_acuracia = tabela_folds["Acuracia_teste"].mean()

desvio_acuracia = tabela_folds["Acuracia_teste"].std()

print(f"Acurácia média: {media_acuracia:.2f}%")

print(f"Desvio padrão: {desvio_acuracia:.2f}%")

print(
    f"Resultado final: {media_acuracia:.2f}% "
    f"± {desvio_acuracia:.2f}%"
)

In [ ]:
contagens_folds = []

for _, linha in tabela_folds.iterrows():

    fold = int(linha["Fold"])
    threshold = linha["Threshold"]

    teste = tabela_completa[
        tabela_completa["Fold"] == fold
    ]

    real = teste["Mesma_pessoa_real"]
    predicao = teste["Similaridade"] >= threshold

    vp = (real & predicao).sum()
    fp = (~real & predicao).sum()
    vn = (~real & ~predicao).sum()
    fn = (real & ~predicao).sum()

    contagens_folds.append({
        "Fold": fold,
        "VP": vp,
        "FP": fp,
        "VN": vn,
        "FN": fn
    })

tabela_contagens = pd.DataFrame(contagens_folds)

display(tabela_contagens)

In [ ]:
tabela_metricas = tabela_contagens.copy()

vp = tabela_metricas["VP"]
fp = tabela_metricas["FP"]
vn = tabela_metricas["VN"]
fn = tabela_metricas["FN"]

tabela_metricas["Precisao"] = vp / (vp + fp) * 100

tabela_metricas["Recall"] = vp / (vp + fn) * 100

tabela_metricas["F1"] = (
    2 * vp / (2 * vp + fp + fn) * 100
)

tabela_metricas["FAR"] = fp / (fp + vn) * 100

tabela_metricas["FRR"] = fn / (fn + vp) * 100

display(tabela_metricas.round(2))

Métrica	O que avalia
Precisão	Quando o sistema diz que é a mesma pessoa, quantas vezes acerta?
Recall	Quantos pares da mesma pessoa o sistema reconhece corretamente?
F1-score	Equilíbrio entre precisão e recall
FAR	Taxa de aceitação incorreta de pessoas diferentes
FRR	Taxa de rejeição incorreta de fotografias da mesma pessoa

In [ ]:
metricas = [
    "Precisao",
    "Recall",
    "F1",
    "FAR",
    "FRR"
]

resumo_metricas = pd.DataFrame({
    "Metrica": ["Acuracia"] + metricas,

    "Media (%)": [
        tabela_folds["Acuracia_teste"].mean()
    ] + [
        tabela_metricas[coluna].mean()
        for coluna in metricas
    ],

    "Desvio padrao (%)": [
        tabela_folds["Acuracia_teste"].std()
    ] + [
        tabela_metricas[coluna].std()
        for coluna in metricas
    ]
})

display(resumo_metricas.round(2))

 a FRR foi maior que a FAR.
Ou seja, o FaceNet rejeitou incorretamente pares da mesma pessoa com mais frequência do que aceitou pares de pessoas diferentes.

In [ ]:
print("SIMILARIDADE COSSENO")
print(tabela_completa["Similaridade"].describe())
#describe:count: quantidade de valores.
#mean: média.
#std: desvio padrão.
#min e max: menor e maior valor.
#25%, 50% e 75%: percentis da distribuição.
print("\nDISTÂNCIA EUCLIDIANA")
print(tabela_completa["Distancia"].describe())

In [ ]:
thresholds_euclidianos = np.arange(0.0, 2.01, 0.01)

resultados_euclidianos = []

for threshold in thresholds_euclidianos:

    real = dados_desenvolvimento["Mesma_pessoa_real"]

    predicao = (
        dados_desenvolvimento["Distancia"] <= threshold
    )

    acuracia = (real == predicao).mean()

    resultados_euclidianos.append({
        "Threshold": round(threshold, 2),
        "Acuracia": acuracia
    })

tabela_euclidiana = pd.DataFrame(resultados_euclidianos)

melhor_linha = tabela_euclidiana.loc[
    tabela_euclidiana["Acuracia"].idxmax()
]

print("Melhor threshold euclidiano:", melhor_linha["Threshold"])

print(
    "Melhor acurácia:",
    round(melhor_linha["Acuracia"] * 100, 2),
    "%"
)

In [ ]:
threshold_euclidiano = 1.12

real = dados_teste["Mesma_pessoa_real"]

predicao_euclidiana = (
    dados_teste["Distancia"] <= threshold_euclidiano
)

acuracia_euclidiana = (
    real == predicao_euclidiana
).mean() * 100

print(
    f"Acurácia euclidiana no fold 1: "
    f"{acuracia_euclidiana:.2f}%"
)

print(
    f"Acurácia cosseno no fold 1: "
    f"{acuracia_teste * 100:.2f}%"
)

Se a distância euclidiana realmente consegue superar o cosseno quando avaliamos fotografias que não participaram da escolha do threshold.
Um detalhe importante: o FaceNet gera embeddings normalizados, então a similaridade cosseno e a distância euclidiana carregam praticamente a mesma informação. Os resultados podem mudar um pouquinho dependendo dos thresholds testados.
Depois vamos automatizar a avaliação euclidiana nos 10 folds, assim como fizemos com o cosseno.

In [ ]:
def encontrar_threshold_euclidiano(dados):
    thresholds = np.arange(0.0, 2.01, 0.01)

    melhor_threshold = None
    melhor_acuracia = -1

    for threshold in thresholds:
        real = dados["Mesma_pessoa_real"]

        predicao = dados["Distancia"] <= threshold

        acuracia = (real == predicao).mean()

        if acuracia > melhor_acuracia:
            melhor_acuracia = acuracia
            melhor_threshold = threshold

    return melhor_threshold, melhor_acuracia

In [ ]:
#teste
threshold_euc, acuracia_euc = (
    encontrar_threshold_euclidiano(dados_desenvolvimento)
)

print("Threshold:", round(threshold_euc, 2))
print("Acurácia:", round(acuracia_euc * 100, 2), "%")

In [ ]:
resultados_euclidianos_folds = []

for fold in range(1, 11):

    # Separar desenvolvimento e teste
    desenvolvimento = tabela_completa[
        tabela_completa["Fold"] != fold
    ]

    teste = tabela_completa[
        tabela_completa["Fold"] == fold
    ]

    # Escolher threshold no desenvolvimento
    threshold, acuracia_dev = (
        encontrar_threshold_euclidiano(desenvolvimento)
    )

    # Avaliar no fold de teste
    real = teste["Mesma_pessoa_real"]
    predicao = teste["Distancia"] <= threshold

    acuracia_teste_euc = (real == predicao).mean()

    # Guardar resultado
    resultados_euclidianos_folds.append({
        "Fold": fold,
        "Threshold": round(threshold, 2),
        "Acuracia_teste": acuracia_teste_euc * 100
    })

tabela_folds_euclidiana = pd.DataFrame(
    resultados_euclidianos_folds
)

display(tabela_folds_euclidiana)

In [ ]:
comparacao = pd.DataFrame({
    "Fold": tabela_folds["Fold"],

    "Cosseno (%)": tabela_folds["Acuracia_teste"],

    "Euclidiana (%)": tabela_folds_euclidiana["Acuracia_teste"]
})

display(comparacao.round(2))

print("\nRESULTADO GERAL")

print(
    "Cosseno:",
    round(comparacao["Cosseno (%)"].mean(), 2),
    "%"
)

print(
    "Euclidiana:",
    round(comparacao["Euclidiana (%)"].mean(), 2),
    "%"
)

In [ ]:
media_euclidiana = tabela_folds_euclidiana[
    "Acuracia_teste"
].mean()

desvio_euclidiana = tabela_folds_euclidiana[
    "Acuracia_teste"
].std()

print(
    f"Euclidiana: {media_euclidiana:.2f}% "
    f"± {desvio_euclidiana:.2f}%"
)

print(
    f"Cosseno: {media_acuracia:.2f}% "
    f"± {desvio_acuracia:.2f}%"
)

In [ ]:
print(tabela_folds_euclidiana.head())

In [ ]:
import pandas as pd

tabela_folds_euclidiana = pd.DataFrame({
    "Fold": list(range(1, 11)),
    "Threshold": [
        1.12, 1.10, 1.10, 1.10, 1.10,
        1.10, 1.08, 1.10, 1.10, 1.10
    ],
    "Acuracia_teste": [
        98.166667, 98.666667, 97.5, 97.5, 99.0,
        98.333333, 97.0, 99.166667, 99.0, 98.666667
    ]
})

media_euclidiana = tabela_folds_euclidiana["Acuracia_teste"].mean()
desvio_euclidiana = tabela_folds_euclidiana["Acuracia_teste"].std()

print(f"Euclidiana: {media_euclidiana:.2f}% ± {desvio_euclidiana:.2f}%")

In [ ]:
from google.colab import drive
import os

drive.mount("/content/drive")

pasta_resultados = "/content/drive/MyDrive/IA_Reconhecimento_Facial/resultados"

os.makedirs(pasta_resultados, exist_ok=True)

tabela_folds_euclidiana.to_csv(
    f"{pasta_resultados}/resultados_euclidiana.csv",
    index=False
)

print("Arquivo salvo com sucesso!")

In [ ]:
tabela_folds = pd.DataFrame({
    "Fold": list(range(1, 11)),
    "Threshold": [0.40] * 10,
    "Acuracia_teste": [
        98.666667,
        98.666667,
        97.500000,
        97.333333,
        99.000000,
        98.166667,
        97.833333,
        99.333333,
        99.166667,
        98.666667
    ]
})

tabela_folds.to_csv(
    f"{pasta_resultados}/resultados_cosseno.csv",
    index=False
)

print("Resultados do cosseno salvos!")

display(tabela_folds)

In [ ]:
resumo_metricas = pd.DataFrame({
    "Metrica": [
        "Acuracia",
        "Precisao",
        "Recall",
        "F1",
        "FAR",
        "FRR"
    ],
    "Media (%)": [
        98.43,
        98.79,
        98.07,
        98.43,
        1.20,
        1.93
    ],
    "Desvio padrao (%)": [
        0.69,
        0.68,
        1.16,
        0.70,
        0.69,
        1.16
    ]
})

resumo_metricas.to_csv(
    f"{pasta_resultados}/metricas_cosseno.csv",
    index=False
)

print("Métricas salvas com sucesso!")

display(resumo_metricas)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))

plt.plot(
    tabela_folds["Fold"],
    tabela_folds["Acuracia_teste"],
    marker="o",
    label="Similaridade cosseno"
)

plt.plot(
    tabela_folds_euclidiana["Fold"],
    tabela_folds_euclidiana["Acuracia_teste"],
    marker="s",
    label="Distância euclidiana"
)

plt.xlabel("Fold")
plt.ylabel("Acurácia (%)")
plt.title("FaceNet: Cosseno vs. Distância Euclidiana")

plt.xticks(range(1, 11))
plt.legend()
plt.grid(True, alpha=0.3)

plt.show()

plt.figure() define o tamanho do gráfico.






plt.plot() desenha uma linha com os resultados de cada fold.



marker="o" coloca círculos nos pontos do cosseno.



marker="s" coloca quadrados nos pontos da distância euclidiana.



plt.legend() identifica as duas linhas.



plt.grid() adiciona uma grade para facilitar a leitura.

In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(
    tabela_folds["Fold"],
    tabela_folds["Acuracia_teste"],
    marker="o",
    label="Similaridade cosseno"
)

plt.plot(
    tabela_folds_euclidiana["Fold"],
    tabela_folds_euclidiana["Acuracia_teste"],
    marker="s",
    label="Distância euclidiana"
)

plt.xlabel("Fold")
plt.ylabel("Acurácia (%)")
plt.title("Comparação das métricas de distância — FaceNet")

plt.xticks(range(1, 11))
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()

plt.savefig(
    f"{pasta_resultados}/comparacao_acuracia.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Gráfico salvo com sucesso!")

In [ ]:
import os

arquivos = os.listdir(pasta_resultados)

print("Arquivos salvos:")

for arquivo in sorted(arquivos):
    print("-", arquivo)

In [ ]:
import pandas as pd
import os

pasta_resultados = "/content/drive/MyDrive/IA_Reconhecimento_Facial/resultados"

tabela_folds = pd.read_csv(
    f"{pasta_resultados}/resultados_cosseno.csv"
)

tabela_folds_euclidiana = pd.read_csv(
    f"{pasta_resultados}/resultados_euclidiana.csv"
)

resumo_metricas = pd.read_csv(
    f"{pasta_resultados}/metricas_cosseno.csv"
)

print("Resultados recuperados com sucesso!")
display(resumo_metricas)

Dados de treinamento: imagens usadas para ajustar os parâmetros do FaceNet.



Dados de teste: imagens usadas apenas para avaliar o desempenho final.

In [ ]:
import os

caminho_lfw = "/root/scikit_learn_data/lfw_home"

for nome in sorted(os.listdir(caminho_lfw)):
    caminho = os.path.join(caminho_lfw, nome)

    if os.path.isdir(caminho):
        print("📁 Pasta:", nome)
    else:
        print("📄 Arquivo:", nome)

In [ ]:
import os

pasta_imagens = "/root/scikit_learn_data/lfw_home/lfw_funneled"

pessoas = [
    nome for nome in os.listdir(pasta_imagens)
    if os.path.isdir(os.path.join(pasta_imagens, nome))
]

total_imagens = 0

for pessoa in pessoas:
    caminho = os.path.join(pasta_imagens, pessoa)

    imagens = [
        arquivo for arquivo in os.listdir(caminho)
        if arquivo.lower().endswith((".jpg", ".jpeg", ".png"))
    ]

    total_imagens += len(imagens)

print("Quantidade de pessoas:", len(pessoas))
print("Quantidade de imagens:", total_imagens)

In [ ]:
#para fazer Para fazer fine-tuning, ajuda ter mais de uma fotografia da mesma pessoa
from collections import Counter

quantidades = []

for pessoa in pessoas:
    caminho = os.path.join(pasta_imagens, pessoa)

    imagens = [
        arquivo for arquivo in os.listdir(caminho)
        if arquivo.lower().endswith((".jpg", ".jpeg", ".png"))
    ]

    quantidades.append(len(imagens))

print("Pessoas com apenas 1 foto:",
      sum(q == 1 for q in quantidades))

print("Pessoas com 2 ou mais fotos:",
      sum(q >= 2 for q in quantidades))

print("Pessoas com 5 ou mais fotos:",
      sum(q >= 5 for q in quantidades))

print("Pessoas com 10 ou mais fotos:",
      sum(q >= 10 for q in quantidades))

In [ ]:
import os

pasta_projeto = "/content/drive/MyDrive/IA_Reconhecimento_Facial"

pastas = [
    "dados_treinamento",
    "modelos",
    "resultados",
    "graficos"
]

for pasta in pastas:
    caminho = os.path.join(pasta_projeto, pasta)
    os.makedirs(caminho, exist_ok=True)
    print("Pasta pronta:", caminho)

dados_treinamento: armazenar as imagens utilizadas no treinamento.



modelos: salvar as versões do FaceNet após o fine-tuning.



resultados: guardar tabelas e métricas dos experimentos.



graficos: guardar as figuras do relatório.

5. Experimento 2 — Fine-Tuning do FaceNet com CelebA

-Investigar se o ajuste fino (fine-tuning) de um modelo FaceNet pré-treinado poderia melhorar sua capacidade de distinguir rostos de pessoas diferentes.
Será utilizado um subconjunto do CelebA, contendo 2.000 imagens faciais de 100 pessoas diferentes.
As imagens foram divididas conforme as partições oficiais do dataset, mantendo identidades diferentes entre treinamento, validação e teste, assim como foi ensinado em sala de aula

In [ ]:
import pandas as pd
import os

caminho_identidades = (
    "/content/drive/MyDrive/IA_Reconhecimento_Facial/"
    "dados_treinamento/identity_CelebA.txt"
)

if os.path.exists(caminho_identidades):
    identidades = pd.read_csv(
        caminho_identidades,
        sep=r"\s+",
        header=None,
        names=["Imagem", "Pessoa"]
    )

    print("Arquivo carregado com sucesso!")
    print("Quantidade de imagens:", len(identidades))
    print("Quantidade de pessoas:", identidades["Pessoa"].nunique()) #conta quantas identidades existem

    display(identidades.head())
else:
    print("Arquivo não encontrado. Verifique o caminho.")

In [ ]:
fotos_por_pessoa = identidades.groupby("Pessoa").size()

print("Pessoas com 2 ou mais fotos:",
      (fotos_por_pessoa >= 2).sum())

print("Pessoas com 10 ou mais fotos:",
      (fotos_por_pessoa >= 10).sum())

print("Pessoas com 20 ou mais fotos:",
      (fotos_por_pessoa >= 20).sum())

print("Pessoas com 50 ou mais fotos:",
      (fotos_por_pessoa >= 50).sum())

print("Maior quantidade de fotos de uma pessoa:",
      fotos_por_pessoa.max())
# Selecionar pessoas com pelo menos 20 fotos
pessoas_elegiveis = fotos_por_pessoa[
    fotos_por_pessoa >= 20
].index

# Selecionar 100 identidades de forma reproduzível
pessoas_selecionadas = (
    pd.Series(pessoas_elegiveis)
    .sample(n=100, random_state=42)
    .tolist()
)

# Selecionar até 20 fotografias por pessoa
subconjunto = (
    identidades[
        identidades["Pessoa"].isin(pessoas_selecionadas)
    ]
    .groupby("Pessoa", group_keys=False)
    .sample(n=20, random_state=42)
)

print("Pessoas selecionadas:",
      subconjunto["Pessoa"].nunique())

print("Fotografias selecionadas:", len(subconjunto))

display(subconjunto.head())

In [ ]:
import os

# Pasta principal do projeto
pasta_projeto = "/content/drive/MyDrive/IA_Reconhecimento_Facial"

# Pasta dos dados utilizados no treinamento
pasta_treinamento = os.path.join(
    pasta_projeto,
    "dados_treinamento"
)

# Verificar se a pasta existe
print("Pasta:", pasta_treinamento)
print("Pasta encontrada:", os.path.isdir(pasta_treinamento))

In [ ]:
import os

caminho_selecao = os.path.join(
    pasta_treinamento,
    "selecao_celebA_2000.csv"
)

subconjunto.to_csv(
    caminho_selecao,
    index=False
)

print("Seleção salva com sucesso!")
print("Local:", caminho_selecao)

In [ ]:
import pandas as pd
import os

caminho_particoes = os.path.join(
    pasta_treinamento,
    "list_eval_partition.txt"
)

particoes = pd.read_csv(
    caminho_particoes,
    sep=r"\s+",
    header=None,
    names=["Imagem", "Particao"]
)

# Juntar a seleção de 2.000 imagens com as partições oficiais
subconjunto_particoes = subconjunto.merge(
    particoes,
    on="Imagem",
    how="left",
    validate="one_to_one"
)

print("Total de imagens:", len(subconjunto_particoes))
print("\nDistribuição das partições:")

print(
    subconjunto_particoes["Particao"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nIdentidades em cada partição:")

print(
    subconjunto_particoes.groupby("Particao")["Pessoa"]
    .nunique()
)

In [ ]:

#pessoa aparece em mais de uma partição.
particoes_por_pessoa = (
    subconjunto_particoes
    .groupby("Pessoa")["Particao"]
    .nunique()
)

pessoas_repetidas = (particoes_por_pessoa > 1).sum()

print("Pessoas em mais de uma partição:", pessoas_repetidas)

if pessoas_repetidas == 0:
    print("OK! Identidades separadas entre as partições.")
else:
    print("Atenção! Existem identidades compartilhadas.")

Logo, terá para treinamento: 75 pessoas e 1500 imagens
validação: 11 pessoas e 220 imagens
teste: 14 pessoas e 280 imagens

In [ ]:
caminho_divisao = os.path.join(
    pasta_treinamento,
    "celebA_divisao_treino_validacao_teste.csv"
)

subconjunto_particoes.to_csv(
    caminho_divisao,
    index=False
)

print("Divisão salva com sucesso!")
print("Arquivo:", caminho_divisao)
import os

caminho_zip = (
    "/content/drive/MyDrive/IA_Reconhecimento_Facial/"
    "dados_treinamento/img_align_celeba.zip"
)

if os.path.isfile(caminho_zip):
    tamanho_gb = os.path.getsize(caminho_zip) / (1024 ** 3)

    print("Arquivo encontrado com sucesso!")
    print(f"Tamanho: {tamanho_gb:.2f} GB")
else:
    print("Arquivo não encontrado!")

In [ ]:
import zipfile

with zipfile.ZipFile(caminho_zip, "r") as zip_ref:
    arquivos_zip = zip_ref.namelist()

    print("Total de arquivos no ZIP:", len(arquivos_zip))
import os
import zipfile
import shutil

# Pasta onde vamos salvar as imagens selecionadas
pasta_imagens = os.path.join(
    pasta_treinamento,
    "imagens_selecionadas"
)

os.makedirs(pasta_imagens, exist_ok=True)

# Nomes das 2.000 fotografias
imagens_desejadas = set(subconjunto_particoes["Imagem"])

extraidas = 0
faltando = []

with zipfile.ZipFile(caminho_zip, "r") as zip_ref:
    for imagem in sorted(imagens_desejadas):

        nome_no_zip = f"img_align_celeba/{imagem}"
        destino = os.path.join(pasta_imagens, imagem)

        if nome_no_zip not in zip_ref.NameToInfo:
            faltando.append(imagem)
            continue

        if not os.path.exists(destino):
            with zip_ref.open(nome_no_zip) as origem:
                with open(destino, "wb") as saida:
                    shutil.copyfileobj(origem, saida)

        extraidas += 1

print("Imagens disponíveis:", extraidas)
print("Imagens não encontradas:", len(faltando))
print("Pasta:", pasta_imagens)

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import os

# Selecionar 6 imagens da nossa base
amostra = subconjunto_particoes.sample(
    n=6,
    random_state=42
)

fig, axes = plt.subplots(2, 3, figsize=(10, 7))

for ax, (_, linha) in zip(axes.flat, amostra.iterrows()):
    caminho = os.path.join(
        pasta_imagens,
        linha["Imagem"]
    )

    imagem = Image.open(caminho).convert("RGB")

    ax.imshow(imagem)
    ax.set_title(f"Pessoa: {linha['Pessoa']}")
    ax.axis("off")

plt.tight_layout()
plt.show()


In [ ]:
from torchvision import transforms

transformacao_facenet = transforms.Compose([
    transforms.Resize((160, 160)),
    transforms.ToTensor(),
    transforms.Lambda(lambda x: (x - 0.5) / 0.5)
])

# Testar a transformação em uma fotografia
nome_imagem = subconjunto_particoes.iloc[0]["Imagem"]

caminho_imagem = os.path.join(
    pasta_imagens,
    nome_imagem
)

imagem = Image.open(caminho_imagem).convert("RGB")

imagem_tensor = transformacao_facenet(imagem)

print("Formato da imagem:", imagem_tensor.shape)
print("Menor valor:", imagem_tensor.min().item())
print("Maior valor:", imagem_tensor.max().item())

In [ ]:
from torch.utils.data import Dataset
from PIL import Image
import os

class CelebADataset(Dataset):

    def __init__(self, dados, pasta_imagens, transformacao):
        self.dados = dados.reset_index(drop=True)
        self.pasta_imagens = pasta_imagens
        self.transformacao = transformacao

    def __len__(self):
        return len(self.dados)

    def __getitem__(self, indice):
        linha = self.dados.iloc[indice]

        caminho = os.path.join(
            self.pasta_imagens,
            linha["Imagem"]
        )

        imagem = Image.open(caminho).convert("RGB")
        imagem = self.transformacao(imagem)

        identidade = int(linha["Pessoa"])

        return imagem, identidade


# Criar o dataset de treinamento
dados_treino = subconjunto_particoes[
    subconjunto_particoes["Particao"] == 0
]

dataset_treino = CelebADataset(
    dados_treino,
    pasta_imagens,
    transformacao_facenet
)

print("Total de imagens de treinamento:", len(dataset_treino))

imagem, identidade = dataset_treino[0]

print("Formato da imagem:", imagem.shape)
print("Identidade:", identidade)

In [ ]:
from torch.utils.data import DataLoader

loader_treino = DataLoader(
    dataset_treino,
    batch_size=16,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

# Testar um lote
imagens, identidades = next(iter(loader_treino))

print("Formato do lote:", imagens.shape)
print("Formato das identidades:", identidades.shape)
print("Quantidade de lotes:", len(loader_treino))
# Identidades presentes no conjunto de treinamento
pessoas_treino = sorted(dados_treino["Pessoa"].unique())

# Criar o mapeamento de identidade para classe
mapa_classes = {
    pessoa: indice
    for indice, pessoa in enumerate(pessoas_treino)
}

# Criar uma cópia com a nova coluna
dados_treino_classes = dados_treino.copy()

dados_treino_classes["Classe"] = (
    dados_treino_classes["Pessoa"].map(mapa_classes)
)

print("Quantidade de classes:", len(mapa_classes))
print("Menor classe:", dados_treino_classes["Classe"].min())
print("Maior classe:", dados_treino_classes["Classe"].max())

display(dados_treino_classes.head())

In [ ]:
class CelebATreinoDataset(CelebADataset):

    def __getitem__(self, indice):
        imagem, _ = super().__getitem__(indice)

        classe = int(self.dados.iloc[indice]["Classe"])

        return imagem, classe


# Criar o Dataset atualizado
dataset_treino_classes = CelebATreinoDataset(
    dados_treino_classes,
    pasta_imagens,
    transformacao_facenet
)

# Atualizar o DataLoader
loader_treino = DataLoader(
    dataset_treino_classes,
    batch_size=16,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

# Conferir os resultados
imagens, classes = next(iter(loader_treino))

print("Formato das imagens:", imagens.shape)
print("Formato das classes:", classes.shape)
print("Menor classe do lote:", classes.min().item())
print("Maior classe do lote:", classes.max().item())
print("Total de classes:", len(mapa_classes))

In [ ]:
import torch
import torch.nn as nn
from facenet_pytorch import InceptionResnetV1

# Selecionar GPU, se disponível
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# Carregar uma nova instância do FaceNet pré-treinado
facenet_treino = InceptionResnetV1(
    pretrained="vggface2",
    classify=False
).to(device)

# Criar a camada de classificação
classificador = nn.Linear(
    512,
    75
).to(device)

print("Dispositivo:", device)
print("FaceNet carregado com sucesso!")
print("Camada de classificação:", classificador)

O FaceNet produz um vetor de 512 características para representar cada rosto. Esse vetor é chamado de embedding.

Nossa nova camada transforma essas 512 características em 75 valores, um para cada identidade do treinamento.

In [ ]:
# Congelar todos os parâmetros do FaceNet
for parametro in facenet_treino.parameters():
    parametro.requires_grad = False

# Liberar as últimas camadas para fine-tuning
for nome, parametro in facenet_treino.named_parameters():
    if nome.startswith(("block8", "last_linear", "last_bn")):
        parametro.requires_grad = True

# Garantir que o classificador será treinado
for parametro in classificador.parameters():
    parametro.requires_grad = True

# Contar parâmetros treináveis
parametros_treinaveis = sum(
    p.numel()
    for p in facenet_treino.parameters()
    if p.requires_grad
)

print("Parâmetros treináveis do FaceNet:", parametros_treinaveis)

print(
    "Parâmetros treináveis do classificador:",
    sum(p.numel() for p in classificador.parameters() if p.requires_grad)
)

In [ ]:
import torch.nn as nn
import torch.optim as optim

# Função de perda para classificação
criterio = nn.CrossEntropyLoss()

# Otimizador com taxas de aprendizado diferentes
otimizador = optim.Adam([
    {
        "params": [
            p for p in facenet_treino.parameters()
            if p.requires_grad
        ],
        "lr": 0.00001
    },
    {
        "params": classificador.parameters(),
        "lr": 0.001
    }
])

print("Função de perda:", criterio)
print("Otimizador: Adam")

for i, grupo in enumerate(otimizador.param_groups):
    print(f"Grupo {i + 1} - Learning rate: {grupo['lr']}")

In [ ]:
import torch.nn as nn

def configurar_modo_treinamento():

    # Ativar treinamento nas camadas liberadas
    facenet_treino.train()

    # Preservar as estatísticas de Batch Normalization
    for modulo in facenet_treino.modules():
        if isinstance(
            modulo,
            nn.modules.batchnorm._BatchNorm
        ):
            modulo.eval()

    # Ativar treinamento no classificador
    classificador.train()


configurar_modo_treinamento()

# Verificar a configuração
bn_treinando = sum(
    1
    for modulo in facenet_treino.modules()
    if isinstance(modulo, nn.modules.batchnorm._BatchNorm)
    and modulo.training
)

print("FaceNet em modo de treinamento:", facenet_treino.training)
print("Classificador em modo de treinamento:", classificador.training)
print("BatchNorms em modo de treinamento:", bn_treinando)

In [ ]:
# Selecionar as imagens de validação
dados_validacao = subconjunto_particoes[
    subconjunto_particoes["Particao"] == 1
].copy()

# Criar o Dataset de validação
dataset_validacao = CelebADataset(
    dados_validacao,
    pasta_imagens,
    transformacao_facenet
)

# Criar o DataLoader de validação
loader_validacao = DataLoader(
    dataset_validacao,
    batch_size=16,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

# Conferir
print("Imagens de validação:", len(dataset_validacao))
print("Pessoas diferentes:", dados_validacao["Pessoa"].nunique())
print("Quantidade de lotes:", len(loader_validacao))

imagens, identidades = next(iter(loader_validacao))

print("Formato das imagens:", imagens.shape)
print("Formato das identidades:", identidades.shape)

In [ ]:
# Pegar um lote de treinamento
imagens, classes = next(iter(loader_treino))

# Enviar para a GPU
imagens = imagens.to(device)
classes = classes.to(device)

# Testar sem atualizar os parâmetros
facenet_treino.eval()
classificador.eval()

with torch.no_grad():
    embeddings = facenet_treino(imagens)
    saidas = classificador(embeddings)
    perda = criterio(saidas, classes)

print("Formato dos embeddings:", embeddings.shape)
print("Formato das saídas:", saidas.shape)
print("Loss inicial:", perda.item())

In [ ]:
from tqdm.auto import tqdm

# Configurar o modo correto de treinamento
configurar_modo_treinamento()

perda_total = 0
acertos = 0
total = 0

for imagens, classes in tqdm(
    loader_treino,
    desc="Treinando época 1"
):

    imagens = imagens.to(device)
    classes = classes.to(device)

    # Zerar os gradientes anteriores
    otimizador.zero_grad()

    # Gerar embeddings
    embeddings = facenet_treino(imagens)

    # Classificar os rostos
    saidas = classificador(embeddings)

    # Calcular a perda
    perda = criterio(saidas, classes)

    # Calcular gradientes
    perda.backward()

    # Atualizar os parâmetros
    otimizador.step()

    # Registrar resultados
    perda_total += perda.item() * imagens.size(0)

    previsoes = saidas.argmax(dim=1)
    acertos += (previsoes == classes).sum().item()
    total += classes.size(0)

loss_media = perda_total / total
acuracia_treino = 100 * acertos / total

print(f"Loss média: {loss_media:.4f}")
print(f"Acurácia de treinamento: {acuracia_treino:.2f}%")

In [ ]:
import os
import torch

pasta_modelos = os.path.join(
    "/content/drive/MyDrive/IA_Reconhecimento_Facial",
    "modelos"
)

os.makedirs(pasta_modelos, exist_ok=True)

caminho_checkpoint = os.path.join(
    pasta_modelos,
    "facenet_epoca_1.pth"
)

torch.save({
    "epoca": 1,
    "facenet": facenet_treino.state_dict(),
    "classificador": classificador.state_dict(),
    "otimizador": otimizador.state_dict(),
    "mapa_classes": mapa_classes,
    "loss_treino": loss_media,
    "acuracia_treino": acuracia_treino
}, caminho_checkpoint)

print("Checkpoint salvo com sucesso!")
print("Arquivo:", caminho_checkpoint)

In [ ]:
# Extrair novamente TODOS os embeddings de validação

facenet_treino.eval()

lista_embeddings = []
lista_identidades = []

with torch.no_grad():
    for imagens, identidades in loader_validacao:

        imagens = imagens.to(device)

        embeddings = facenet_treino(imagens)

        embeddings = F.normalize(
            embeddings, p=2, dim=1
        )

        lista_embeddings.append(
            embeddings.cpu().numpy()
        )

        lista_identidades.append(
            identidades.numpy()
        )

# Reunir todos os lotes
embeddings_validacao = np.concatenate(
    lista_embeddings, axis=0
)

identidades_validacao = np.concatenate(
    lista_identidades, axis=0
)

print("Imagens processadas:", len(embeddings_validacao))
print("Formato dos embeddings:", embeddings_validacao.shape)
print("Pessoas diferentes:", len(np.unique(identidades_validacao)))

assert len(embeddings_validacao) == len(dataset_validacao), \
    "Nem todas as imagens foram processadas!"

In [ ]:
import numpy as np

# Calcular a similaridade do cosseno
matriz_similaridade = (
    embeddings_validacao @ embeddings_validacao.T
)

# Selecionar pares únicos de imagens
indices_i, indices_j = np.triu_indices(
    len(identidades_validacao),
    k=1
)

similaridades = matriz_similaridade[
    indices_i, indices_j
]

# Verificar se as imagens pertencem à mesma pessoa
mesma_pessoa = (
    identidades_validacao[indices_i]
    == identidades_validacao[indices_j]
)

# Separar os pares
similaridades_positivas = similaridades[mesma_pessoa]
similaridades_negativas = similaridades[~mesma_pessoa]

# Mostrar resultados
print("Pares positivos:", len(similaridades_positivas))
print("Pares negativos:", len(similaridades_negativas))

print(
    "Similaridade média positiva:",
    round(float(similaridades_positivas.mean()), 4)
)

print(
    "Similaridade média negativa:",
    round(float(similaridades_negativas.mean()), 4)
)

In [ ]:
from sklearn.metrics import roc_auc_score

# Rótulos reais: 1 = mesma pessoa, 0 = pessoas diferentes
rotulos_reais = mesma_pessoa.astype(int)

# Calcular AUC
auc_validacao = roc_auc_score(
    rotulos_reais,
    similaridades
)

print(f"AUC de validação: {auc_validacao:.4f}")
print(f"AUC em porcentagem: {auc_validacao * 100:.2f}%")

In [ ]:
from facenet_pytorch import InceptionResnetV1

# Carregar o FaceNet original, sem fine-tuning
facenet_original = InceptionResnetV1(
    pretrained="vggface2",
    classify=False
).to(device)

# Modo de avaliação
facenet_original.eval()

# Extrair embeddings das mesmas imagens de validação
lista_embeddings_originais = []

with torch.no_grad():

    for imagens, identidades in loader_validacao:

        imagens = imagens.to(device)

        embeddings = facenet_original(imagens)

        embeddings = F.normalize(
            embeddings,
            p=2,
            dim=1
        )

        lista_embeddings_originais.append(
            embeddings.cpu().numpy()
        )

embeddings_originais = np.concatenate(
    lista_embeddings_originais,
    axis=0
)

print(
    "Formato dos embeddings originais:",
    embeddings_originais.shape
)

In [ ]:
from sklearn.metrics import roc_auc_score
import numpy as np

# Similaridade do cosseno do FaceNet original
matriz_original = (
    embeddings_originais @ embeddings_originais.T
)

# Usar exatamente os mesmos pares da avaliação anterior
similaridades_originais = matriz_original[
    indices_i, indices_j
]

# Calcular a AUC do modelo original
auc_original = roc_auc_score(
    rotulos_reais,
    similaridades_originais
)

# Comparar os dois modelos
print(f"FaceNet original: {auc_original * 100:.2f}%")
print(f"FaceNet com fine-tuning: {auc_validacao * 100:.2f}%")

diferenca = (auc_validacao - auc_original) * 100

print(f"Diferença: {diferenca:+.2f} pontos percentuais")

Modelo	AUC no teste
FaceNet original	99,47%
FaceNet com fine-tuning — época 1	99,51%
O FaceNet original e o modelo ajustado apresentaram resultados muito próximos.
O treinamento adicional não demonstrou uma melhoria consistente no conjunto de teste utilizado.


In [ ]:
import pandas as pd

historico_treinamento = pd.DataFrame({
    "Epoca": [0, 1],
    "Modelo": [
        "FaceNet original",
        "FaceNet fine-tuning"
    ],
    "AUC_validacao": [
        auc_original,
        auc_validacao
    ],
    "Loss_treino": [
        None,
        loss_media
    ],
    "Acuracia_treino": [
        None,
        acuracia_treino
    ]
})

display(historico_treinamento)

# Salvar no Google Drive
caminho_historico = os.path.join(
    pasta_modelos,
    "historico_treinamento.csv"
)

historico_treinamento.to_csv(
    caminho_historico,
    index=False
)

print("Histórico salvo com sucesso!")

In [ ]:
from tqdm.auto import tqdm

# Preparar o modelo para continuar o treinamento
configurar_modo_treinamento()

perda_total = 0
acertos = 0
total = 0

for imagens, classes in tqdm(
    loader_treino,
    desc="Treinando época 2"
):
    imagens = imagens.to(device)
    classes = classes.to(device)

    otimizador.zero_grad()

    embeddings = facenet_treino(imagens)
    saidas = classificador(embeddings)

    perda = criterio(saidas, classes)

    perda.backward()
    otimizador.step()

    perda_total += perda.item() * imagens.size(0)

    previsoes = saidas.argmax(dim=1)
    acertos += (previsoes == classes).sum().item()
    total += classes.size(0)

loss_epoca_2 = perda_total / total
acuracia_epoca_2 = 100 * acertos / total

print(f"Loss média da época 2: {loss_epoca_2:.4f}")
print(f"Acurácia de treinamento: {acuracia_epoca_2:.2f}%")

In [ ]:
# Avaliar o modelo após a segunda época
facenet_treino.eval()

lista_embeddings_epoca_2 = []
lista_identidades_epoca_2 = []

with torch.no_grad():
    for imagens, identidades in loader_validacao:

        imagens = imagens.to(device)

        embeddings = facenet_treino(imagens)
        embeddings = F.normalize(embeddings, p=2, dim=1)

        lista_embeddings_epoca_2.append(
            embeddings.cpu().numpy()
        )

        lista_identidades_epoca_2.append(
            identidades.numpy()
        )

embeddings_epoca_2 = np.concatenate(
    lista_embeddings_epoca_2, axis=0
)

identidades_epoca_2 = np.concatenate(
    lista_identidades_epoca_2, axis=0
)

# Garantir que a ordem das imagens não mudou
assert np.array_equal(
    identidades_epoca_2,
    identidades_validacao
)

# Calcular similaridades dos mesmos pares
matriz_epoca_2 = (
    embeddings_epoca_2 @ embeddings_epoca_2.T
)

similaridades_epoca_2 = matriz_epoca_2[
    indices_i, indices_j
]

# Calcular AUC
auc_epoca_2 = roc_auc_score(
    rotulos_reais,
    similaridades_epoca_2
)

print(f"FaceNet original: {auc_original * 100:.2f}%")
print(f"Fine-tuning época 1: {auc_validacao * 100:.2f}%")
print(f"Fine-tuning época 2: {auc_epoca_2 * 100:.2f}%")

In [ ]:
# Salvar checkpoint da época 2
caminho_epoca_2 = os.path.join(
    pasta_modelos,
    "facenet_epoca_2.pth"
)

torch.save({
    "epoca": 2,
    "facenet": facenet_treino.state_dict(),
    "classificador": classificador.state_dict(),
    "otimizador": otimizador.state_dict(),
    "mapa_classes": mapa_classes,
    "loss_treino": loss_epoca_2,
    "acuracia_treino": acuracia_epoca_2,
    "auc_validacao": auc_epoca_2
}, caminho_epoca_2)

# Atualizar histórico
historico_treinamento.loc[
    len(historico_treinamento)
] = [
    2,
    "FaceNet fine-tuning",
    auc_epoca_2,
    loss_epoca_2,
    acuracia_epoca_2
]

historico_treinamento.to_csv(
    caminho_historico,
    index=False
)

display(historico_treinamento)

print("Época 2 salva com sucesso!")

In [ ]:
# Selecionar as imagens reservadas para teste
dados_teste = subconjunto_particoes[
    subconjunto_particoes["Particao"] == 2
].copy()

# Criar o Dataset
dataset_teste = CelebADataset(
    dados_teste,
    pasta_imagens,
    transformacao_facenet
)

# Criar o DataLoader
loader_teste = DataLoader(
    dataset_teste,
    batch_size=16,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

# Conferir os dados
print("Imagens de teste:", len(dataset_teste))
print("Pessoas diferentes:", dados_teste["Pessoa"].nunique())
print("Quantidade de lotes:", len(loader_teste))

# Verificar se as identidades são realmente separadas
pessoas_teste = set(dados_teste["Pessoa"])
pessoas_treinamento = set(dados_treino["Pessoa"])
pessoas_validacao = set(dados_validacao["Pessoa"])

print(
    "Pessoas compartilhadas com treinamento:",
    len(pessoas_teste & pessoas_treinamento)
)

print(
    "Pessoas compartilhadas com validação:",
    len(pessoas_teste & pessoas_validacao)
)

In [ ]:
# Carregar o checkpoint da melhor época
checkpoint = torch.load(
    caminho_checkpoint,
    map_location=device,
    weights_only=False
)

# Restaurar os pesos do FaceNet
facenet_treino.load_state_dict(
    checkpoint["facenet"]
)

# Restaurar os pesos do classificador
classificador.load_state_dict(
    checkpoint["classificador"]
)

# Preparar para avaliação
facenet_treino.eval()
classificador.eval()

print("Checkpoint recuperado com sucesso!")
print("Época carregada:", checkpoint["epoca"])
print("Modelo em avaliação:", not facenet_treino.training)

In [ ]:
# Colocar os dois modelos em modo de avaliação
facenet_original.eval()
facenet_treino.eval()

lista_original = []
lista_ajustado = []
lista_ids_teste = []

with torch.no_grad():

    for imagens, identidades in loader_teste:

        imagens = imagens.to(device)

        # FaceNet original
        emb_original = facenet_original(imagens)
        emb_original = F.normalize(
            emb_original, p=2, dim=1
        )

        # FaceNet com fine-tuning
        emb_ajustado = facenet_treino(imagens)
        emb_ajustado = F.normalize(
            emb_ajustado, p=2, dim=1
        )

        lista_original.append(
            emb_original.cpu().numpy()
        )

        lista_ajustado.append(
            emb_ajustado.cpu().numpy()
        )

        lista_ids_teste.append(
            identidades.numpy()
        )

# Reunir todos os lotes
embeddings_teste_original = np.concatenate(
    lista_original, axis=0
)

embeddings_teste_ajustado = np.concatenate(
    lista_ajustado, axis=0
)

identidades_teste = np.concatenate(
    lista_ids_teste, axis=0
)

# Conferir
print(
    "FaceNet original:",
    embeddings_teste_original.shape
)

print(
    "FaceNet ajustado:",
    embeddings_teste_ajustado.shape
)

print(
    "Pessoas diferentes:",
    len(np.unique(identidades_teste))
)

In [ ]:
from sklearn.metrics import roc_auc_score
import numpy as np

# Selecionar pares únicos, sem repetir fotografias
i_teste, j_teste = np.triu_indices(
    len(identidades_teste),
    k=1
)

# Identificar pares da mesma pessoa
rotulos_teste = (
    identidades_teste[i_teste]
    == identidades_teste[j_teste]
).astype(int)

# Calcular similaridades do modelo original
similaridades_teste_original = (
    embeddings_teste_original
    @ embeddings_teste_original.T
)[i_teste, j_teste]

# Calcular similaridades do modelo ajustado
similaridades_teste_ajustado = (
    embeddings_teste_ajustado
    @ embeddings_teste_ajustado.T
)[i_teste, j_teste]

# Calcular AUC dos dois modelos
auc_teste_original = roc_auc_score(
    rotulos_teste,
    similaridades_teste_original
)

auc_teste_ajustado = roc_auc_score(
    rotulos_teste,
    similaridades_teste_ajustado
)

# Mostrar resultados
print("Pares positivos:", int(rotulos_teste.sum()))
print("Pares negativos:", int((rotulos_teste == 0).sum()))

print(f"FaceNet original: {auc_teste_original * 100:.2f}%")
print(f"FaceNet ajustado: {auc_teste_ajustado * 100:.2f}%")

diferenca_teste = (
    auc_teste_ajustado - auc_teste_original
) * 100

print(f"Diferença: {diferenca_teste:+.2f} pontos percentuais")

o fine-tuning não trouxe uma melhoria consistente no reconhecimento facial.


*  O FaceNet original já tinha um desempenho excelente.
*  A primeira época de fine-tuning trouxe uma melhoria muito pequena na validação.
* Essa melhoria não se manteve no teste.
* A segunda época aumentou a acurácia de treinamento, mas reduziu a AUC de validação.



In [ ]:
# Criar tabela com os resultados finais
comparacao_final = pd.DataFrame({
    "Modelo": [
        "FaceNet original",
        "FaceNet fine-tuning - época 1"
    ],
    "AUC_validacao": [
        auc_original,
        auc_validacao
    ],
    "AUC_teste": [
        auc_teste_original,
        auc_teste_ajustado
    ]
})

# Exibir tabela
display(comparacao_final)

# Salvar no Google Drive
caminho_comparacao = os.path.join(
    pasta_modelos,
    "comparacao_final_facenet.csv"
)

comparacao_final.to_csv(
    caminho_comparacao,
    index=False
)

print("Comparação final salva com sucesso!")

* A avaliação do FaceNet original com o LFW.
* O fine-tuning com o CelebA.
* A comparação entre o modelo original e o ajustado.
* A avaliação final com pessoas que não participaram do treinamento.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os

modelos = ["FaceNet original", "FaceNet ajustado"]

auc_val = [auc_original * 100, auc_validacao * 100]
auc_teste = [auc_teste_original * 100, auc_teste_ajustado * 100]

x = np.arange(len(modelos))
largura = 0.35

fig, ax = plt.subplots(figsize=(9, 5))

barras_val = ax.bar(
    x - largura / 2,
    auc_val,
    largura,
    label="Validação"
)

barras_teste = ax.bar(
    x + largura / 2,
    auc_teste,
    largura,
    label="Teste"
)

ax.set_ylabel("AUC (%)")
ax.set_title("Comparação do FaceNet antes e após fine-tuning")
ax.set_xticks(x)
ax.set_xticklabels(modelos)
ax.set_ylim(0, 105)
ax.legend()

ax.bar_label(barras_val, fmt="%.2f%%", padding=3)
ax.bar_label(barras_teste, fmt="%.2f%%", padding=3)

plt.tight_layout()

caminho_grafico = os.path.join(
    pasta_modelos,
    "comparacao_auc_facenet.png"
)

plt.savefig(caminho_grafico, dpi=300, bbox_inches="tight")
plt.show()

print("Gráfico salvo com sucesso!")

In [ ]:
import pandas as pd
import os

# Carregar os resultados já salvos
resultados_cosseno = pd.read_csv(
    os.path.join(pasta_resultados, "resultados_cosseno.csv")
)

resultados_euclidiana = pd.read_csv(
    os.path.join(pasta_resultados, "resultados_euclidiana.csv")
)

metricas_cosseno = pd.read_csv(
    os.path.join(pasta_resultados, "metricas_cosseno.csv")
)

# Mostrar a estrutura dos arquivos
print("=== RESULTADOS COSSENO ===")
print(resultados_cosseno.columns.tolist())
display(resultados_cosseno.head())

print("\n=== RESULTADOS EUCLIDIANA ===")
print(resultados_euclidiana.columns.tolist())
display(resultados_euclidiana.head())

print("\n=== MÉTRICAS COSSENO ===")
print(metricas_cosseno.columns.tolist())
display(metricas_cosseno.head())

In [ ]:
# Calcular acurácia média e desvio-padrão
media_cosseno = resultados_cosseno["Acuracia_teste"].mean()
desvio_cosseno = resultados_cosseno["Acuracia_teste"].std()

media_euclidiana = resultados_euclidiana["Acuracia_teste"].mean()
desvio_euclidiana = resultados_euclidiana["Acuracia_teste"].std()

# Criar tabela comparativa
tabela_lfw = pd.DataFrame({
    "Método": [
        "Similaridade do cosseno",
        "Distância euclidiana"
    ],
    "Acurácia média (%)": [
        media_cosseno,
        media_euclidiana
    ],
    "Desvio-padrão (%)": [
        desvio_cosseno,
        desvio_euclidiana
    ]
})

display(tabela_lfw.round(2))

# Mostrar também as outras métricas do cosseno
print("\nMétricas adicionais do FaceNet:")
display(metricas_cosseno)

In [ ]:
import matplotlib.pyplot as plt
import os

# Selecionar as métricas principais
metricas_principais = metricas_cosseno[
    metricas_cosseno["Metrica"].isin([
        "Acuracia",
        "Precisao",
        "Recall",
        "F1"
    ])
].copy()

fig, ax = plt.subplots(figsize=(9, 5))

barras = ax.bar(
    metricas_principais["Metrica"],
    metricas_principais["Media (%)"]
)

ax.set_title("Desempenho do FaceNet no LFW")
ax.set_ylabel("Resultado médio (%)")
ax.set_ylim(0, 105)

ax.bar_label(barras, fmt="%.2f%%", padding=3)

plt.tight_layout()

caminho_grafico_metricas = os.path.join(
    pasta_resultados,
    "metricas_facenet_lfw.png"
)

plt.savefig(
    caminho_grafico_metricas,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Gráfico das métricas salvo com sucesso!")

Nosso experimento mostrou que o FaceNet pré-treinado apresentou desempenho elevado na verificação facial com o LFW.

Também investigamos se o fine-tuning com um subconjunto do CelebA poderia melhorar a representação dos rostos. Embora a primeira época tenha apresentado uma pequena vantagem na validação, essa vantagem não se manteve no teste.

In [ ]:
print("=== EXPERIMENTO 1: LFW ===")

print(
    f"Acurácia média - Cosseno: "
    f"{cosseno['Acuracia_teste'].mean():.2f}%"
)

print(
    f"Acurácia média - Euclidiana: "
    f"{euclidiana['Acuracia_teste'].mean():.2f}%"
)

print("\n=== EXPERIMENTO 2: CELEBA ===")

for _, linha in celeba.iterrows():
    print(
        f"{linha['Modelo']}: "
        f"AUC teste = {linha['AUC_teste'] * 100:.2f}%"
    )

diferenca = (
    celeba.loc[1, "AUC_teste"]
    - celeba.loc[0, "AUC_teste"]
) * 100

print(f"\nDiferença de AUC: {diferenca:+.4f} ponto percentual")

In [ ]:
import pandas as pd
import os

# Carregar os resultados do LFW
cosseno = pd.read_csv(
    os.path.join(pasta_resultados, "resultados_cosseno.csv")
)

euclidiana = pd.read_csv(
    os.path.join(pasta_resultados, "resultados_euclidiana.csv")
)

# Carregar os resultados do CelebA
celeba = pd.read_csv(
    os.path.join(pasta_modelos, "comparacao_final_facenet.csv")
)

print("=== EXPERIMENTO 1: LFW ===")

print(
    f"Acurácia média - Cosseno: "
    f"{cosseno['Acuracia_teste'].mean():.2f}%"
)

print(
    f"Acurácia média - Euclidiana: "
    f"{euclidiana['Acuracia_teste'].mean():.2f}%"
)

print("\n=== EXPERIMENTO 2: CELEBA ===")

display(celeba)

In [ ]:
import pandas as pd
import os

pasta = "/content/drive/MyDrive/IA_Reconhecimento_Facial/resultados"

cosseno = pd.read_csv(
    os.path.join(pasta, "resultados_cosseno.csv")
)

euclidiana = pd.read_csv(
    os.path.join(pasta, "resultados_euclidiana.csv")
)

comparacao = pd.DataFrame({
    "Métrica": ["Cosseno", "Euclidiana"],
    "Acurácia média (%)": [
        cosseno["Acuracia_teste"].mean(),
        euclidiana["Acuracia_teste"].mean()
    ],
    "Desvio padrão (%)": [
        cosseno["Acuracia_teste"].std(),
        euclidiana["Acuracia_teste"].std()
    ]
})

display(comparacao.round(2))

In [ ]:
import pandas as pd

caminho = "/content/drive/MyDrive/IA_Reconhecimento_Facial/modelos/comparacao_final_facenet.csv"

tabela = pd.read_csv(caminho)
display(tabela)

### Comparação entre similaridade do cosseno e distância euclidiana

Foram avaliadas duas métricas para comparar os embeddings
gerados pelo FaceNet: similaridade do cosseno e distância
euclidiana.

Ambas foram avaliadas no conjunto LFW utilizando validação
cruzada com 10 folds.

| Métrica | Acurácia média | Desvio padrão |
|---|---|---|
| Similaridade do cosseno | 98,43% | 0,69% |
| Distância euclidiana | 98,30% | 0,74% |

A similaridade do cosseno apresentou acurácia média
0,13 ponto percentual superior à distância euclidiana.

Como a  diferença foi pequena, e os resultados
indicam que ambas as métricas apresentaram desempenho
semelhante na tarefa de verificação facial.

## 6. Conclusão Geral

Este projeto teve como objetivo avaliar o desempenho do modelo FaceNet na tarefa de verificação facial e investigar os efeitos do fine-tuning utilizando diferentes conjuntos de imagens.

No primeiro experimento, realizado com o dataset LFW, o FaceNet pré-treinado apresentou resultados elevados, alcançando acurácia média de **98,43% com similaridade do cosseno** e **98,30% com distância euclidiana**, utilizando validação cruzada com 10 folds.

No segundo experimento, foi realizado o fine-tuning do FaceNet utilizando 2.000 imagens do dataset CelebA, distribuídas entre treinamento, validação e teste, sem compartilhamento de identidades entre os conjuntos.

Na avaliação final com o CelebA, foram utilizadas 280 imagens de 14 identidades, formando 39.060 pares de comparação, sendo 2.660 positivos e 36.400 negativos. O modelo FaceNet original obteve AUC de **99,47%**, enquanto o modelo ajustado alcançou **99,51%**. Embora tenha ocorrido uma pequena melhoria numérica após o fine-tuning, a diferença de aproximadamente **0,03 ponto percentual** não é suficiente, isoladamente, para demonstrar um ganho estatisticamente significativo.

Assim, os resultados indicam que o FaceNet pré-treinado já apresentava desempenho elevado no conjunto avaliado, enquanto o treinamento adicional produziu uma alteração muito pequena no desempenho de verificação facial.

É importante destacar que os experimentos utilizaram conjuntos de dados e métricas de avaliação diferentes. Portanto, os valores de acurácia obtidos no LFW não devem ser comparados diretamente aos valores de AUC obtidos no CelebA.

### 6.1. Limitações

Algumas limitações devem ser consideradas:

- O fine-tuning foi realizado com um subconjunto relativamente pequeno do CelebA.
- O treinamento foi limitado a duas épocas.
- Foram avaliadas configurações específicas de arquitetura, taxa de aprendizado e estratégia de ajuste.
- A avaliação final no CelebA utilizou imagens de apenas 14 identidades não presentes no treinamento.
- As comparações entre pares de imagens não são totalmente independentes, pois uma mesma imagem pode participar de vários pares.

### 6.2. Trabalhos Futuros

Como possibilidades de continuidade, destacam-se:

- Investigar diferentes quantidades de imagens e identidades no treinamento.
- Testar outras estratégias de fine-tuning.
- Avaliar o modelo ajustado em outros conjuntos de dados.
- Investigar o impacto de condições como iluminação, pose e qualidade das imagens.
- Comparar outras arquiteturas de reconhecimento facial.

### 6.3. Considerações Finais

Os experimentos demonstraram a capacidade do FaceNet pré-treinado de produzir representações faciais adequadas para a verificação de identidade nos conjuntos avaliados.

Além dos resultados quantitativos, o projeto permitiu explorar etapas importantes

## 7. Experimento com Dataset Privado

### 7.1. Objetivo

Avaliar a similaridade facial produzida pelo FaceNet em um pequeno conjunto de fotografias próprias, investigando possíveis variações associadas às expressões faciais.

### 7.2. Dataset

Foram utilizadas 6 fotografias minhas pessoa, capturadas com webcam e armazenadas localmente.

As imagens não foram incluídas no notebook nem disponibilizadas publicamente.

### 7.3. Metodologia

- Detecção facial utilizando MTCNN.
- Extração de embeddings utilizando FaceNet pré-treinado em VGGFace2.
- Cálculo da similaridade de cosseno entre todos os pares de fotografias.
- Total de 15 comparações entre imagens da mesma pessoa.
- Processamento realizado localmente, em CPU.

### 7.4. Resultados

O MTCNN detectou corretamente um rosto em cada uma das 6 fotografias.

O FaceNet gerou 6 embeddings faciais, permitindo realizar 15 comparações entre pares de imagens da mesma pessoa.

Entre os resultados observados:

- Maior similaridade: 0,9043 (fotografias 1 e 3).
- Menor similaridade: 0,4714 (fotografias 4 e 6).

### 7.5. Discussão

Foi observado que a fotografia 6, capturada com uma expressão facial exagerada (careta), apresentou algumas das menores similaridades.

Esse resultado sugere que alterações na expressão facial podem influenciar os embeddings gerados pelo FaceNet.

Entretanto, como o experimento utilizou apenas uma pessoa e não controlou todas as condições de captura, não é possível estabelecer uma relação causal entre a expressão facial e a redução da similaridade.

O experimento também não permite calcular a acurácia de verificação facial, pois não foram utilizados pares de pessoas diferentes.

### 7.6. Privacidade

As fotografias foram processadas localmente e não foram inseridas no notebook público ou no repositório do projeto.

Os embeddings foram utilizados apenas durante a execução do programa, sem salvamento em arquivos.